# 04 - Modelado: Clustering de Clientes con K-Means

## 🎯 Objetivo
Agrupar clientes según su comportamiento de compra utilizando el algoritmo K-Means.

Trabajamos sobre la tabla `gold_customer_features`, que contiene las variables:
- **Recency** - Qué tan reciente compra el cliente
- **Frequency** - Qué tan seguido compra  
- **Monetary** - Cuánto dinero gasta

In [0]:

df = spark.table("bigdata.default.gold_customer_features")
display(df)

## ⚠️ Parte 2: Escalado de Datos (CLAVE)

Antes de aplicar K-Means, realizamos un paso crítico: el **escalado de variables** usando `StandardScaler`.

### ¿Por qué es necesario?
Las variables están en **diferentes escalas**:
- `Recency` está en **días**
- `Frequency` es un **conteo** (números pequeños)
- `Monetary` está en **valores monetarios** (números mucho más grandes)

### ⚡ Problema si no escalamos:
El algoritmo daría más peso a `Monetary`, **distorsionando los clústeres**.

### ✅ Solución:
El escalado transforma los datos para que todas las variables tengan:
- **Media = 0**
- **Desviación estándar = 1**

Esto asegura que cada variable contribuya equitativamente al modelo.

In [0]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

# Convertir a Pandas
pdf = df.select("id_cliente", "recency", "frequency", "monetary").toPandas()

# Escalar
scaler = StandardScaler()
pdf[["recency_scaled", "frequency_scaled", "monetary_scaled"]] = scaler.fit_transform(
    pdf[["recency", "frequency", "monetary"]]
)

print(pdf.head())

## 🤖 Parte 3: Aplicación de K-Means

Una vez escalados los datos, aplicamos el algoritmo **K-Means**.

### ¿Cómo funciona K-Means?

1. **Definir K** - Se elige el número de clústeres
2. **Inicializar centroides** - Se colocan aleatoriamente en el espacio
3. **Asignar puntos** - Cada cliente se asigna al centroide más cercano
4. **Recalcular centroides** - Se mueven al centro de los puntos asignados
5. **Repetir** - El proceso itera hasta converger

### 🎯 Resultado:
Cada cliente queda asignado a un **clúster según similitud en su comportamiento** de compra.

In [0]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42)
pdf["cluster"] = kmeans.fit_predict(
    pdf[["recency_scaled", "frequency_scaled", "monetary_scaled"]]
)

print(pdf["cluster"].value_counts())

## ❓ Parte 4: Determinar el K Óptimo

Para seleccionar el número correcto de clústeres, ejecutamos el notebook **`05_evaluation_mlflow.ipynb`** que evalúa:
- **Método del Codo** (Inercia)
- **Silhouette Score**

> ⚠️ **Nota:** El K=4 usado aquí fue seleccionado tras analizar ambas métricas en el notebook 05.

### Riesgos de elegir K incorrecto:

| Problema | K muy pequeño | K muy grande |
|----------|---------------|--------------|
| Resultado | Segmentos muy generales | Segmentación excesiva |
| Valor de negocio | Poca accionable | Sin diferenciación real |

In [0]:
from pyspark.ml.clustering import KMeans
import json

# Leer K óptimo generado por notebook 05 (si existe), sino usar default
try:
    with open("/tmp/kmeans_config.json", "r") as f:
        k_config = json.load(f)
        optimal_k = k_config["optimal_k"]
        print(f"📂 K óptimo cargado desde evaluación: {optimal_k}")
except:
    optimal_k = 4  # Default basado en análisis previo
    print(f"⚠️ Usando K default: {optimal_k}")

# Entrenar modelo con K óptimo
kmeans = KMeans(
    k=optimal_k,
    seed=42,
    featuresCol="features",
    predictionCol="cluster"
)

model = kmeans.fit(df_scaled)
df_clustered = model.transform(df_scaled)

print(f"✅ Modelo entrenado con K={optimal_k}")
print(f"📊 Inercia: {model.summary.trainingCost:,.0f}")

## 📝 Parte 5: Guardado de Resultados

Guardamos la tabla final con los segmentos asignados en la capa **Gold** para consumo por BI y equipos de negocio.

La tabla `gold_customer_segments` contiene:
- `CustomerID` - Identificador del cliente
- `Recency`, `Frequency`, `Monetary` - Métricas RFM
- `cluster` - Segmento asignado (0, 1, 2, o 3)

In [0]:
# Mostrar centroides de cada clúster (perfil de segmentos)
centroids = model.clusterCenters()
print("📍 Centroides de los clústeres:")
for i, centroid in enumerate(centroids):
    print(f"\n🔷 Cluster {i}:")
    print(f"   Recency:   {centroid[0]:.3f}")
    print(f"   Frequency: {centroid[1]:.3f}")
    print(f"   Monetary:  {centroid[2]:.3f}")

In [0]:
# Visualizar asignación de clientes a clústeres
display(df_clustered.select("CustomerID", "Recency", "Frequency", "Monetary", "cluster"))

In [0]:
# Guardar resultados en tabla Delta
# Usamos mode overwrite para permitir reejecuciones durante desarrollo
df_clustered.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_customer_segments")

print("✅ Tabla 'gold_customer_segments' guardada exitosamente")
print(f"📊 Total clientes segmentados: {df_clustered.count()}")

# Guardar metadatos del modelo para trazabilidad
model_metadata = {
    "k": optimal_k,
    "inertia": model.summary.trainingCost,
    "total_customers": df_clustered.count(),
    "features": ["Recency", "Frequency", "Monetary"]
}
with open("/tmp/model_metadata.json", "w") as f:
    json.dump(model_metadata, f)
print(f"💾 Metadatos del modelo guardados")

In [0]:
## ✅ Conclusión del Notebook 04

### Flujo Completado:
1. ✅ Lectura de datos RFM desde `gold_customer_features`
2. ✅ Escalado con StandardScaler (media=0, std=1)
3. ✅ Entrenamiento K-Means con K=4 (óptimo determinado en notebook 05)
4. ✅ Análisis de centroides
5. ✅ Guardado en `gold_customer_segments`

### Próximo Paso:
El equipo de BI puede consumir la tabla `gold_customer_segments` para crear dashboards de segmentación de clientes.

### Flujo completo de la arquitectura:
```
Bronze (raw) → Silver (limpia) → Gold (RFM) → Gold (Segmentos)
     ↑                                              ↓
  online_retail                            gold_customer_segments
```